# PlantVillage: exploration + training (EfficientNet-B0), one notebook
Kaggle: set **Accelerator = GPU** and **Internet = On**. Attach `emmarex/plantdisease` or let the notebook download it.
Run all cells top to bottom.

Outputs in `/kaggle/working`: `plant_disease_effnetb0.pth`, `labels.json`, `model_meta.json`, `metrics.json`,
`confusion_matrix.png`, `training_curves.png`, `gradcam_samples.png`.

---
# Part 1: Data exploration

In [ ]:
import os, random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_ROOT = "/kaggle/input/plantdisease"
EXT = {".jpg", ".jpeg", ".png"}


def scan(root):
    rows, seen = [], set()
    for dp, dn, fn in os.walk(root):
        dn.sort()
        imgs = sorted(f for f in fn if Path(f).suffix.lower() in EXT)
        if not imgs:
            continue
        cls = os.path.basename(dp)
        if cls in seen:          # skip nested duplicate copy of the class folders
            continue
        seen.add(cls)
        rows += [(os.path.join(dp, f), cls) for f in imgs]
    return rows


rows = scan(DATA_ROOT)
if not rows:
    DATA_ROOT = "/kaggle/input"
    rows = scan(DATA_ROOT)
if not rows:
    print("Dataset not attached here -> downloading emmarex/plantdisease with kagglehub (needs Internet ON)...")
    try:
        import kagglehub
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kagglehub"])
        import kagglehub
    DATA_ROOT = kagglehub.dataset_download("emmarex/plantdisease")
    rows = scan(DATA_ROOT)
if not rows:
    raise RuntimeError("No images found. Attach emmarex/plantdisease via Add Input, or run kagglehub.login() and re-run.")

print("root used:", DATA_ROOT)
df = pd.DataFrame(rows, columns=["path", "label"])
df["crop"] = df.label.str.split("_").str[0]
classes = sorted(df.label.unique())
print("DataFrame shape (rows, cols):", df.shape)
print("number of classes:", len(classes))
print("number of crops:", df.crop.nunique(), sorted(df.crop.unique()))
df.head()

## Class distribution

In [ ]:
counts = df.label.value_counts()
print(counts.to_string())
print(f"\nlargest / smallest = {counts.max()} / {counts.min()} = {counts.max() / counts.min():.1f}x")

fig, ax = plt.subplots(figsize=(9, 5))
counts.sort_values().plot.barh(ax=ax, color="#4c9f70")
ax.set_xlabel("images"); ax.set_title("Images per class")
for i, v in enumerate(counts.sort_values().values):
    ax.text(v + 20, i, str(v), va="center", fontsize=8)
plt.tight_layout(); plt.show()

## Image shape and size statistics

In [ ]:
# one image: the raw array shape the model will see before resizing
img0 = Image.open(df.path.iloc[0])
a0 = np.array(img0.convert("RGB"))
print("example file :", df.path.iloc[0])
print("PIL size (W,H):", img0.size, "| mode:", img0.mode)
print("numpy shape (H,W,C):", a0.shape, "| dtype:", a0.dtype, "| min/max:", a0.min(), a0.max())

# size / mode statistics on a random sample (reading all 20k headers is slow-ish but fine; sample 1500)
sample = df.sample(min(1500, len(df)), random_state=SEED)
sizes, modes = Counter(), Counter()
for p in sample.path:
    with Image.open(p) as im:
        sizes[im.size] += 1
        modes[im.mode] += 1
print("\nimage sizes (W,H) in sample:", dict(sizes.most_common(5)))
print("color modes in sample:", dict(modes))

In [ ]:
# tensor shapes as the training pipeline will produce them
try:
    import torch
    from torchvision import transforms
    tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
    batch = torch.stack([tf(Image.open(p).convert("RGB")) for p in df.path.sample(8, random_state=SEED)])
    print("one batch tensor shape (B, C, H, W):", tuple(batch.shape), "| dtype:", batch.dtype)
    print("pixel range after ToTensor:", float(batch.min()), "to", float(batch.max()))
except ImportError:
    print("torch/torchvision not installed; skipping tensor shape")
print("\nfull dataset as one array would be:", (len(df), 224, 224, 3), "(N, H, W, C) at 224x224")

## Sample images: one per class

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, c in zip(axes.ravel(), classes):
    p = df[df.label == c].sample(1, random_state=SEED).path.iloc[0]
    im = Image.open(p).convert("RGB")
    ax.imshow(im); ax.axis("off")
    ax.set_title(f"{c}\n{im.size[0]}x{im.size[1]}", fontsize=8)
for ax in axes.ravel()[len(classes):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## Random sample (16 images) with labels

In [ ]:
rs = df.sample(16, random_state=SEED + 1)
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for ax, (_, r) in zip(axes.ravel(), rs.iterrows()):
    ax.imshow(Image.open(r.path).convert("RGB")); ax.axis("off")
    ax.set_title(r.label, fontsize=7)
plt.tight_layout(); plt.show()

## Several images from one class (change `CLASS` to look closer)

In [ ]:
CLASS = "Tomato_Late_blight"
if CLASS not in classes:
    CLASS = classes[0]
sub = df[df.label == CLASS].sample(8, random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, p in zip(axes.ravel(), sub.path):
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
fig.suptitle(CLASS)
plt.tight_layout(); plt.show()

---
# Part 2: Training
## Prepare labels and drop exact duplicates

In [ ]:
import hashlib
df = df.copy()
df["md5"] = df.path.map(lambda p: hashlib.md5(open(p, "rb").read()).hexdigest())
before = len(df)
df = df.drop_duplicates("md5").reset_index(drop=True)
print("exact duplicates removed:", before - len(df), "| images now:", len(df))

classes = sorted(df.label.unique())
cls2idx = {c: i for i, c in enumerate(classes)}
df["y"] = df.label.map(cls2idx)
NUM_CLASSES = len(classes)
assert NUM_CLASSES == 15, f"expected 15 classes, got {NUM_CLASSES}"

## Imports and config

In [ ]:
import os, json, random, hashlib, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content/outputs"
os.makedirs(OUT, exist_ok=True)
IMG = 224
BS = 64
EPOCHS = 8
LR = 3e-4
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_cuda = device.type == "cuda"
print("device:", device)

## 2. Stratified split (70 / 15 / 15)
Note: PlantVillage contains many shots of the same leaf, so near-duplicates can still land in different splits
and make test accuracy optimistic. Mention this in the report.

In [ ]:
train_df, tmp_df = train_test_split(df, test_size=0.30, stratify=df.y, random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, stratify=tmp_df.y, random_state=SEED)
print(len(train_df), len(val_df), len(test_df))

train_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


class LeafDS(Dataset):
    def __init__(self, d, tf):
        self.p, self.y, self.tf = d.path.tolist(), d.y.tolist(), tf

    def __len__(self):
        return len(self.p)

    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]


nw = min(4, os.cpu_count() or 2)
train_dl = DataLoader(LeafDS(train_df, train_tf), BS, shuffle=True, num_workers=nw, pin_memory=use_cuda, drop_last=True)
val_dl = DataLoader(LeafDS(val_df, eval_tf), BS, shuffle=False, num_workers=nw, pin_memory=use_cuda)
test_dl = DataLoader(LeafDS(test_df, eval_tf), BS, shuffle=False, num_workers=nw, pin_memory=use_cuda)

# class weights (inverse frequency) to offset the imbalance
counts = train_df.y.value_counts().sort_index().values
cw = torch.tensor(len(train_df) / (NUM_CLASSES * counts), dtype=torch.float32).to(device)

## 3. Model and training

In [ ]:
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
in_f = model.classifier[1].in_features
model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, NUM_CLASSES))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_dl))
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)


@torch.no_grad()
def evaluate(dl):
    model.eval()
    loss_sum, preds, ys = 0.0, [], []
    for x, y in dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=use_cuda):
            out = model(x)
            loss_sum += criterion(out, y).item() * len(y)
        preds += out.argmax(1).cpu().tolist()
        ys += y.cpu().tolist()
    return loss_sum / len(ys), np.array(preds), np.array(ys)


best_f1, hist = 0.0, []
ckpt = f"{OUT}/plant_disease_effnetb0.pth"
for ep in range(1, EPOCHS + 1):
    model.train()
    t0, run = time.time(), 0.0
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_cuda):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        run += loss.item()
    vl, vp, vy = evaluate(val_dl)
    acc, f1 = (vp == vy).mean(), f1_score(vy, vp, average="macro")
    hist.append(dict(epoch=ep, train_loss=run / len(train_dl), val_loss=vl, val_acc=acc, val_f1=f1))
    flag = ""
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), ckpt)
        flag = "  <- saved"
    print(f"ep {ep}/{EPOCHS} | train {run/len(train_dl):.4f} | val loss {vl:.4f} acc {acc:.4f} macroF1 {f1:.4f} | {time.time()-t0:.0f}s{flag}")

In [ ]:
h = pd.DataFrame(hist)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val"); ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_acc, label="acc"); ax[1].plot(h.epoch, h.val_f1, label="macro-F1"); ax[1].set_title("validation"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/training_curves.png", dpi=150); plt.show()

## 4. Test-set evaluation (best checkpoint)

In [ ]:
model.load_state_dict(torch.load(ckpt, map_location=device))
_, tp, ty = evaluate(test_dl)
test_acc = float((tp == ty).mean())
test_f1 = float(f1_score(ty, tp, average="macro"))
print(f"TEST acc {test_acc:.4f} | macro-F1 {test_f1:.4f}\n")
print(classification_report(ty, tp, target_names=classes, digits=4))

cm = confusion_matrix(ty, tp)
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(classes, rotation=90, fontsize=7); ax.set_yticklabels(classes, fontsize=7)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6, color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("Test confusion matrix")
plt.tight_layout(); plt.savefig(f"{OUT}/confusion_matrix.png", dpi=150); plt.show()

## 5. Grad-CAM (optional, for the explainability section of the report)

In [ ]:
def gradcam(model, x, target=None):
    feats, grads = {}, {}
    layer = model.features[-1]

    def fwd(m, i, o):
        feats["a"] = o
        o.register_hook(lambda g: grads.__setitem__("g", g))

    h = layer.register_forward_hook(fwd)
    model.eval(); model.zero_grad()
    out = model(x)
    c = int(out.argmax(1)) if target is None else target
    out[0, c].backward()
    h.remove()
    w = grads["g"].mean((2, 3), keepdim=True)
    cam = F.relu((w * feats["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam.detach().cpu().numpy(), c


samples = test_df.sample(6, random_state=SEED)
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
for k, (_, r) in enumerate(samples.iterrows()):
    img = Image.open(r.path).convert("RGB").resize((IMG, IMG))
    x = eval_tf(img).unsqueeze(0).to(device)
    cam, c = gradcam(model, x)
    ax[0, k].imshow(img); ax[0, k].set_title(f"true: {r.label[:22]}", fontsize=7); ax[0, k].axis("off")
    ax[1, k].imshow(img); ax[1, k].imshow(cam, cmap="jet", alpha=0.45); ax[1, k].set_title(f"pred: {classes[c][:22]}", fontsize=7); ax[1, k].axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/gradcam_samples.png", dpi=150); plt.show()

## 6. Export for deployment

In [ ]:
with open(f"{OUT}/labels.json", "w") as f:
    json.dump(classes, f, indent=2)
with open(f"{OUT}/model_meta.json", "w") as f:
    json.dump({"arch": "efficientnet_b0", "img_size": IMG, "mean": MEAN, "std": STD, "num_classes": NUM_CLASSES}, f, indent=2)
with open(f"{OUT}/metrics.json", "w") as f:
    json.dump({"test_acc": test_acc, "test_macro_f1": test_f1, "best_val_macro_f1": float(best_f1)}, f, indent=2)

# the checkpoint is a state_dict; make sure it reloads on CPU (this is what the web app will do)
m2 = models.efficientnet_b0(weights=None)
m2.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m2.classifier[1].in_features, NUM_CLASSES))
m2.load_state_dict(torch.load(ckpt, map_location="cpu"))
m2.eval()
print("CPU reload OK | size MB:", round(os.path.getsize(ckpt) / 1e6, 1))
print(os.listdir(OUT))

## Save everything (zip always; copies to Google Drive when running on Colab)

In [ ]:
import os, shutil, zipfile
FILES = ["plant_disease_effnetb0.pth", "labels.json", "model_meta.json", "metrics.json",
         "confusion_matrix.png", "training_curves.png", "gradcam_samples.png"]
FILES = [f for f in FILES if os.path.exists(f"{OUT}/{f}")]
print("found:", FILES)

zip_name = "plant_disease_artifacts.zip"
with zipfile.ZipFile(f"{OUT}/{zip_name}", "w", zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        z.write(f"{OUT}/{f}", f)
print("zip:", f"{OUT}/{zip_name}", round(os.path.getsize(f"{OUT}/{zip_name}") / 1e6, 1), "MB")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/plant_disease_project"
    os.makedirs(dst, exist_ok=True)
    for f in FILES + [zip_name]:
        shutil.copy(f"{OUT}/{f}", dst)
    print("copied to Google Drive:", dst, os.listdir(dst))
except Exception as e:
    print("Not on Colab (Kaggle cannot mount Drive):", type(e).__name__)
    print("Download", zip_name, "from the Output panel, then upload it to Drive by hand.")